In [1]:
import os
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer # Import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# TASK 1 & DATA INGESTION

def load_and_preprocess_raw():
    df = pd.read_csv("/content/googleplaystore.csv")
    df = df.drop_duplicates()
    # Drop rows where 'Rating' is NaN immediately after loading and dropping duplicates
    df.dropna(subset=['Rating'], inplace=True)
    return df

### Regenerating Data After Rating NaN Fix

To ensure that `y_train` is free of `NaN` values, we need to re-run the `load_and_preprocess_raw` function (which now includes `df.dropna(subset=['Rating'], inplace=True)`) and then re-create our feature (`X`) and target (`y`) dataframes. After this, we will re-perform the `train_test_split` and re-fit the `ColumnTransformer`.

In [2]:
# Re-load and preprocess the raw data using the updated function
df = load_and_preprocess_raw()

# Re-apply cleaning for numeric features
df['Reviews'] = pd.to_numeric(df['Reviews'], errors='coerce')

def clean_size(val):
    if pd.isna(val) or val == 'Varies with device':
        return np.nan
    val = str(val).upper().strip()
    if 'M' in val:
        return float(val.replace('M', '')) * 1024
    if 'K' in val:
        return float(val.replace('K', ''))
    return np.nan

df['Size'] = df['Size'].apply(clean_size)
df['Installs'] = df['Installs'].astype(str).str.replace('+', '', regex=False).str.replace(',', '', regex=False)
df['Installs'] = pd.to_numeric(df['Installs'], errors='coerce')
df['Price'] = df['Price'].astype(str).str.replace('$', '', regex=False)
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')

# Define Features (X) and Target (y) again from the newly cleaned df
feature_cols = ['Category', 'Content Rating', 'Type', 'Reviews', 'Size', 'Installs', 'Price']
X = df[feature_cols]
y = df['Rating']

print(f"New Features shape: {X.shape}, New Target shape: {y.shape}")
print(f"Number of null values in new y: {y.isnull().sum()}")

New Features shape: (8893, 7), New Target shape: (8893,)
Number of null values in new y: 0


Now that `X` and `y` are regenerated from the fully cleaned `df`, we can proceed with re-running the `train_test_split`.

In [3]:
# Re-run the train_test_split with the updated X and y
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f"Training samples: {X_train.shape[0]} | Testing samples: {X_test.shape[0]}")
print(f"Number of null values in y_train after re-split: {y_train.isnull().sum()}")

Training samples: 7114 | Testing samples: 1779
Number of null values in y_train after re-split: 0


Finally, we need to re-fit the `ColumnTransformer` (preprocessor) on the newly generated `X_train` to ensure all transformations are consistent with the clean data.

In [4]:
# Re-fit the preprocessor on the new X_train

# Identify categorical and numerical columns (assuming they are still the same)
categorical_cols = ['Category', 'Content Rating', 'Type']
numeric_cols = ['Reviews', 'Size', 'Installs', 'Price']

# Set up preprocessing transformers (as defined previously)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ]
)

# Fit preprocessor on X_train ONLY, then transform both
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

print("Preprocessor re-fitted and data transformed successfully.")

Preprocessor re-fitted and data transformed successfully.


In [6]:
print(f"dataset shape: {df.shape}")
print("\n---DataFrame Info ---")
df.info()

dataset shape: (8893, 13)

---DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
Index: 8893 entries, 0 to 10840
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   App             8893 non-null   object 
 1   Category        8893 non-null   object 
 2   Rating          8893 non-null   float64
 3   Reviews         8892 non-null   float64
 4   Size            7424 non-null   float64
 5   Installs        8892 non-null   float64
 6   Type            8893 non-null   object 
 7   Price           8892 non-null   float64
 8   Content Rating  8892 non-null   object 
 9   Genres          8893 non-null   object 
 10  Last Updated    8893 non-null   object 
 11  Current Ver     8889 non-null   object 
 12  Android Ver     8890 non-null   object 
dtypes: float64(5), object(8)
memory usage: 972.7+ KB


In [7]:
# Remove corrupt row if present
if '1.9' in df['Category'].values:
    df = df[df['Category'] != '1.9']

In [8]:
df = load_and_preprocess_raw()

# Clean numeric features
df['Reviews'] = pd.to_numeric(df['Reviews'], errors='coerce')

def clean_size(val):
    if pd.isna(val) or val == 'Varies with device':
        return np.nan
    val = str(val).upper().strip()
    if 'M' in val:
        return float(val.replace('M', '')) * 1024
    if 'K' in val:
        return float(val.replace('K', ''))
    return np.nan

df['Size'] = df['Size'].apply(clean_size)
df['Installs'] = df['Installs'].astype(str).str.replace('+', '', regex=False).str.replace(',', '', regex=False)
df['Installs'] = pd.to_numeric(df['Installs'], errors='coerce')
df['Price'] = df['Price'].astype(str).str.replace('$', '', regex=False)
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')

# Define Features (X) and Target (y)
feature_cols = ['Category', 'Content Rating', 'Type', 'Reviews', 'Size', 'Installs', 'Price']
X = df[feature_cols]
y = df['Rating']

print(f"Features shape: {X.shape}, Target shape: {y.shape}")

Features shape: (8893, 7), Target shape: (8893,)


In [9]:
# TASK 2: TRAIN/TEST SPLIT FIRST & PREPROCESSING CONFIGURATION
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f"Training samples: {X_train.shape[0]} | Testing samples: {X_test.shape[0]}")

Training samples: 7114 | Testing samples: 1779


In [10]:
from sklearn.impute import SimpleImputer

# Identify categorical and numerical columns
categorical_cols = ['Category', 'Content Rating', 'Type']
numeric_cols = ['Reviews', 'Size', 'Installs', 'Price']

# Set up preprocessing transformers
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ]
)

# Fit preprocessor on X_train ONLY, then transform both
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

In [11]:
# TASK 3: TRAIN & COMPARE REQUIRED MODELS
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree Regressor": DecisionTreeRegressor(random_state=42),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=100, random_state=42)
}

results = []

print("\n--- Model Baseline Evaluation (Primary Metric: R²) ---")
for name, model in models.items():
    model.fit(X_train_prep, y_train)
    y_pred = model.predict(X_test_prep)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results.append({
        "Model": name,
        "MAE": round(mae, 4),
        "RMSE": round(rmse, 4),
        "R2_Score": round(r2, 4)
    })

results_df = pd.DataFrame(results).sort_values(by="R2_Score", ascending=False)
print(results_df.to_string(index=False))

# Best model selection based on primary metric (R²)
best_model_name = results_df.iloc[0]["Model"]
print(f"\nBest Performing Model on Test Set: {best_model_name}")


--- Model Baseline Evaluation (Primary Metric: R²) ---
                  Model    MAE   RMSE  R2_Score
Random Forest Regressor 0.3200 0.4992    0.1231
      Linear Regression 0.3633 0.5259    0.0267
Decision Tree Regressor 0.4200 0.6741   -0.5992

Best Performing Model on Test Set: Random Forest Regressor


In [12]:
# TASK 4: IMPROVE AND VALIDATE VIA PIPELINE
# 1. Wrap Preprocessor and Best Model in a single scikit-learn Pipeline
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42))
])

# 2. Run 5-Fold Cross-Validation on the full pipeline
print("\n--- Running 5-Fold Cross-Validation ---")
cv_results = cross_validate(
    full_pipeline, X_train, y_train,
    cv=5,
    scoring='r2',
    return_train_score=False
)

cv_scores = cv_results['test_score']
print(f"5-Fold R² Scores: {np.round(cv_scores, 4)}")
print(f"Mean CV R² Score: {np.mean(cv_scores):.4f} (+/- {np.std(cv_scores):.4f})")

# 3. Hyperparameter Search using GridSearchCV
print("\n--- Running GridSearchCV Hyperparameter Tuning ---")
param_grid = {
    'regressor__n_estimators': [50, 100],
    'regressor__max_depth': [10, 20, None],
    'regressor__min_samples_split': [2, 5]
}

grid_search = GridSearchCV(
    estimator=full_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print(f"Best Hyperparameters: {grid_search.best_params_}")
print(f"Best CV R² Score from GridSearch: {grid_search.best_score_:.4f}")



--- Running 5-Fold Cross-Validation ---
5-Fold R² Scores: [0.1268 0.0689 0.1033 0.12   0.0076]
Mean CV R² Score: 0.0853 (+/- 0.0437)

--- Running GridSearchCV Hyperparameter Tuning ---
Best Hyperparameters: {'regressor__max_depth': 10, 'regressor__min_samples_split': 5, 'regressor__n_estimators': 100}
Best CV R² Score from GridSearch: 0.1244


In [13]:
# Evaluate tuned pipeline on unseen test data
tuned_test_pred = grid_search.best_estimator_.predict(X_test)
tuned_r2 = r2_score(y_test, tuned_test_pred)
tuned_mae = mean_absolute_error(y_test, tuned_test_pred)
tuned_rmse = np.sqrt(mean_squared_error(y_test, tuned_test_pred))

print(f"\n--- Final Tuned Model Test Set Performance ---")
print(f"Tuned Test R²:   {tuned_r2:.4f}")
print(f"Tuned Test MAE:  {tuned_mae:.4f}")
print(f"Tuned Test RMSE: {tuned_rmse:.4f}")


--- Final Tuned Model Test Set Performance ---
Tuned Test R²:   0.1524
Tuned Test MAE:  0.3267
Tuned Test RMSE: 0.4908
